# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MEDOXIII/FlyRank-Internship-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MEDOXIII/FlyRank-Internship-ML"
REPO_DIR = "FlyRank-Internship-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

Working dir: D:\Projects\FlyRank-Internship-ML\FlyRank-Internship-ML


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I chose Ranking/scoring, over classification because it answer   'which N pages does the team open first this week'  and not 'will this page decline' (yes/no),and that   given limited review capacity. That needs an ordered priority score, not a label.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

review_capacity_per_week = 50
print(f"pool size: {len(df):,} pages")
print(f"a {review_capacity_per_week}-page/week review queue covers {review_capacity_per_week / len(df):.3%} of the pool")
print("-> too small a slice to treat every page as a yes/no classification target; ranking picks the top N instead")

pool size: 30,000 pages
a 50-page/week review queue covers 0.167% of the pool
-> too small a slice to treat every page as a yes/no classification target; ranking picks the top N instead


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I would initially use the starter dataset's is_declining_label as the target, where is_declining_label = trend_direction == "down". This is a defined proxy label based on the current 90-day snapshot, not a future observed outcome. The starter CSV does not contain a future time window, so a target such as decline over the next 30 days cannot be built from it alone. For a stronger future-looking version, I would use the warehouse daily performance data to construct a leakage-safe future target. I will also exclude trend_direction and trend_pct from the feature set to avoid leakage.

In [3]:
label = (df["trend_direction"] == "down").astype(int)
print("is_declining_label rate:", round(label.mean(), 4))

day_counts = {"days_since_last_update", "days_with_impressions",
              "days_with_sessions", "content_age_days"}
date_like_cols = [c for c in df.columns
                  if ("date" in c.lower() or "time" in c.lower()) and c not in day_counts]
print("columns naming an absolute date/timestamp:", date_like_cols or "none")
print("-> no way to check whether a page actually declined after the snapshot;")
print("   the label is defined from the within-window trend, not observed in a later period")

is_declining_label rate: 0.5421
columns naming an absolute date/timestamp: none
-> no way to check whether a page actually declined after the snapshot;
   the label is defined from the within-window trend, not observed in a later period


## 3. Success metric

*One metric you can defend. What number means 'good'?*

One metric I can defend is Precision@50, because the goal is to rank a limited number of pages for review. Since 54.2% of the starter pages are already labeled as declining, a naive strategy that treats every page as positive gives a Precision@50 baseline of about 0.54. Therefore, I would consider a ranking approach useful only if it can consistently achieve precision@50 meaningfully above 0.54 and outperform the simple rule-based baselines.

In [4]:
import sys
import numpy as np
sys.path.insert(0, "scripts")
from ml_utils import precision_at_k

label = (df["trend_direction"] == "down").astype(int)
print("base rate:", round(label.mean(), 4))

rng = np.random.default_rng(42)
random_p50 = [precision_at_k(label.values, rng.random(len(label)), 50) for _ in range(200)]
print("mean precision@50 with a random/no-signal ranking (200 trials):", round(np.mean(random_p50), 4))
print("-> matches the 0.54 baseline used above; a ranking model only earns its keep by beating this")

base rate: 0.5421


mean precision@50 with a random/no-signal ranking (200 trials): 0.5338
-> matches the 0.54 baseline used above; a ranking model only earns its keep by beating this


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Each row represents one unique content page (content_id) for a client. I will treat the content page as the decision unit because the final output is a ranked review queue where each item is a page that a reviewer can act on.

In [5]:
print(f"rows x cols: {df.shape[0]:,} x {df.shape[1]}")
print("duplicate content_id rows:", df["content_id"].duplicated().sum())
print("content_ids attached to >1 client:", (df.groupby("content_id")["client_id"].nunique() > 1).sum())

lane_slice = df[["content_id", "client_id", "trend_direction", "ctr", "avg_position", "content_age_days"]]
lane_slice.head()

rows x cols: 30,000 x 44
duplicate content_id rows: 0
content_ids attached to >1 client: 0


,content_id,client_id,trend_direction,ctr,avg_position,content_age_days
0,content_304f48230142,client_f369cb89fc,down,0.76,10.6,187
1,content_a1fb4e703a9e,client_4e07408562,down,0.05,20.3,445
2,content_9aa793d4d895,client_7f2253d7e2,down,0.09,36.5,141
3,content_331d6c4de07b,client_19581e27de,stable,0.49,6.2,463
4,content_d99b7a2d90ca,client_3fdba35f04,down,0.13,44.0,263


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The pattern is hard to capture with a single if-statement. Most numeric signals, including CTR, average position, and engagement rate, have correlations close to zero with the decline label. A simple CTR-based rule reaches Precision@50 = 0.58, but this is within one standard deviation of random-ranking noise, so it does not represent a meaningful improvement over the 0.542 base-rate benchmark. One exception is content_age_days, which has a stronger correlation of −0.164, and a youngest-content-first ranking reaches Precision@50 = 0.66. However, this is still a modest single-signal rule: in 500 seeded random-ranking trials, 28 trials (5.6%) matched or exceeded 0.66. This suggests that there may be useful signal in the data, but it is unlikely to be captured reliably by one simple threshold and should be tested through combinations of signals.

In [6]:
import sys
import numpy as np
sys.path.insert(0, "scripts")
from ml_utils import precision_at_k

label = (df["trend_direction"] == "down").astype(int)

numeric_cols = ["ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
                "impressions_90d", "sessions_90d", "word_count", "content_age_days",
                "days_since_last_update"]
corrs = df[numeric_cols].corrwith(label).sort_values(key=abs, ascending=False)
print("correlation with is_declining_label:")
print(corrs.round(3))

rule_ctr = -df["ctr"].fillna(0)
rule_age = -df["content_age_days"]
p_ctr = precision_at_k(label.values, rule_ctr.values, 50)
p_age = precision_at_k(label.values, rule_age.values, 50)
print("\nsimple single-feature rules vs the 0.542 base rate:")
print("rule: lowest ctr first           -> precision@50:", p_ctr)
print("rule: youngest content_age first -> precision@50:", p_age)

rng = np.random.default_rng(0)
random_trials = np.array([precision_at_k(label.values, rng.random(len(label)), 50) for _ in range(500)])
mean, std = random_trials.mean(), random_trials.std()
print(f"\nrandom-order precision@50 over 500 seeded trials: mean={mean:.3f}, std={std:.3f}")

z_ctr = (p_ctr - mean) / std
z_age = (p_age - mean) / std
print(f"ctr rule is {z_ctr:.2f} std from random noise -> not a meaningful improvement")
print(f"age rule is {z_age:.2f} std from random noise")

hits_age = int((random_trials >= p_age).sum())
print(f"random trials that matched or exceeded the age rule's {p_age}: {hits_age}/{len(random_trials)} "
      f"({hits_age / len(random_trials):.1%}) -> a real but modest, noisy lift")

correlation with is_declining_label:
content_age_days         -0.164
word_count                0.090
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
sessions_90d             -0.023
impressions_90d          -0.018
engagement_rate          -0.013
scroll_rate              -0.003
ai_traffic_pct            0.002
dtype: float64



simple single-feature rules vs the 0.542 base rate:
rule: lowest ctr first           -> precision@50: 0.58
rule: youngest content_age first -> precision@50: 0.66



random-order precision@50 over 500 seeded trials: mean=0.540, std=0.069
ctr rule is 0.57 std from random noise -> not a meaningful improvement
age rule is 1.73 std from random noise
random trials that matched or exceeded the age rule's 0.66: 28/500 (5.6%) -> a real but modest, noisy lift


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.